# Computer Engineering Department ML Project SOP
## Week 4: Model Evaluation
**Task List:** Test the model on the test dataset and compute metrics. Check for overfitting or underfitting.

---
### 1. Comprehensive Test Set Metrics
Accuracy alone is insufficient on imbalanced datasets. We evaluate Precision, Recall, F1-Score, ROC-AUC, Specificity, and Confusion Matrix.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Load data
df = pd.read_csv('../data/Loan_default.csv').drop(columns=['LoanID'])
binary_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']
for col in binary_cols:
    df[col] = df[col].apply(lambda x: 1 if str(x).strip().lower() in ['yes', '1', 'true'] else 0)
df = pd.get_dummies(df, columns=['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose'], drop_first=False)

X = df.drop(columns=['Default']).values
y = df['Default'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit model
clf = DecisionTreeClassifier(max_depth=7, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=["Non-Default (0)", "Default (1)"]))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")


### 2. Confusion Matrix Analysis
Examining False Positives and False Negatives:


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: Repaid (0)', 'Pred: Default (1)'],
            yticklabels=['Actual: Repaid (0)', 'Actual: Default (1)'])
plt.title("Decision Tree Confusion Matrix")
plt.ylabel("Ground Truth")
plt.xlabel("Predicted Class")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (Correctly Repaid): {tn:,}")
print(f"False Positives (Rejected Creditworthy): {fp:,}")
print(f"False Negatives (Approved Defaulter): {fn:,}")
print(f"True Positives (Correctly Flagged Default): {tp:,}")


### 3. Diagnosing Overfitting vs Underfitting (Bias-Variance Tradeoff)
We sweep the `max_depth` parameter from 1 to 16, measuring both training and testing accuracy to diagnose:
- **Underfitting (High Bias):** Shallow depth (< 4) fails to capture feature interactions.
- **Optimal Complexity:** Depth 6 to 8 achieves maximum generalization.
- **Overfitting (High Variance):** Deep trees (> 12) memorize training noise with declining test accuracy.


In [ ]:
depths = list(range(1, 17))
train_scores = []
test_scores = []

# Subsample for fast interactive diagnostic run
X_sub, _, y_sub, _ = train_test_split(X_train, y_train, train_size=25000, random_state=42, stratify=y_train)

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_sub, y_sub)
    train_scores.append(accuracy_score(y_sub, m.predict(X_sub)))
    test_scores.append(accuracy_score(y_test, m.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(depths, train_scores, marker='o', label="Training Set Accuracy", color="#2b5c8f")
plt.plot(depths, test_scores, marker='s', label="Test Set Accuracy", color="#d95f02")
plt.axvline(x=7, color='grey', linestyle=':', label="Optimal Depth ~ 7")
plt.xlabel("Max Tree Depth")
plt.ylabel("Accuracy Score")
plt.title("Bias-Variance Diagnostics: Overfitting vs Underfitting")
plt.legend()
plt.show()
